# Analysis of QS 2025 Dataset: Which Region Has the Strongest Education System?

This notebook analyzes the QS 2025 university rankings dataset to determine which world region exhibits the strongest education system. We load the data, compute region-level education strength metrics, and visualize the results.

**Prerequisites**: Run `START_HERE.ipynb` first to install dependencies.

Steps:
1. Load required libraries and the dataset
2. Define a metric for "education system strength" using available indicators
3. Aggregate metrics by region
4. Visualize and report the top region(s)

In [ ]:
import os, sys, subprocess, pathlib

# Find repo root (works on both Brev and Docker)
repo_root = None
_result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True)
if _result.returncode == 0:
    repo_root = _result.stdout.strip()
if not repo_root:
    _p = pathlib.Path.cwd()
    for _dir in [_p] + list(_p.parents):
        if (_dir / 'pyproject.toml').exists():
            repo_root = str(_dir)
            break
if not repo_root:
    for _c in [pathlib.Path.home() / 'data-explorer-launchable', pathlib.Path('/app')]:
        if (_c / 'pyproject.toml').exists():
            repo_root = str(_c)
            break
if not repo_root:
    raise RuntimeError('Could not find repo root')

os.chdir(repo_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Load dataset (latin1 encoding needed for accented characters in institution names)
csv_path = os.path.join(repo_root, 'sample_data', 'QS_2025.csv')
df = pd.read_csv(csv_path, encoding='latin1')
print(f'Loaded dataset: {df.shape[0]} universities, {df.shape[1]} columns')
df.head()

In [ ]:
# Identify score columns and compute region-level education strength
score_cols = [c for c in df.columns if c.endswith('_Score')]
print(f'Score columns ({len(score_cols)}): {score_cols}')

# Convert score columns to numeric (coerce errors from rank-style entries like "15=")
for c in score_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Summary statistics
display(df[score_cols].describe())

# Compute region-level aggregates
region_scores = df.groupby('Region')[score_cols].mean()
region_counts = df.groupby('Region').size().rename('Institution_Count')

# Composite education-strength metric: mean across all score indicators
region_scores['Composite_Education_Strength'] = region_scores.mean(axis=1)
region_summary = region_scores[['Composite_Education_Strength']].join(region_counts)
region_summary = region_summary.sort_values('Composite_Education_Strength', ascending=False)

print('\nRegion summary:')
display(region_summary)

# Bar plot
plt.figure(figsize=(10, 6))
region_summary['Composite_Education_Strength'].plot(kind='bar', color='C0')
plt.ylabel('Composite Education Strength (mean of score indicators)')
plt.title('Composite Education Strength by Region (QS 2025)')
plt.tight_layout()
plt.show()

In [ ]:
# Enhanced visualization with value labels
rs = region_summary['Composite_Education_Strength']

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(rs)))
bars = ax.bar(rs.index, rs.values, color=colors, edgecolor='black')

ax.set_ylabel('Composite Education Strength (mean of score indicators)', fontsize=12)
ax.set_title('Composite Education Strength by Region (QS 2025)', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(60, rs.max() * 1.15))
ax.tick_params(axis='x', rotation=30, labelsize=11)
ax.tick_params(axis='y', labelsize=11)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 6),
                textcoords='offset points',
                ha='center', va='bottom', fontsize=10, fontweight='semibold')

plt.tight_layout()
plt.show()

top_region = rs.idxmax()
top_value = rs.max()
print(f'Top region by composite education strength: {top_region} ({top_value:.2f})')
print(f'\nFull region summary:')
display(region_summary)